# MNIST 손글씨 숫자 분류 프로젝트

## 1. IDX 포맷 데이터 로딩 (이미지 + 라벨)

In [1]:
import numpy as np
import struct

def load_images(path):
    with open(path, 'rb') as f:
        _, num, rows, cols = struct.unpack(">IIII", f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)

def load_labels(path):
    with open(path, 'rb') as f:
        _, num = struct.unpack(">II", f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

X_train = load_images("train-images.idx3-ubyte")
y_train = load_labels("train-labels.idx1-ubyte")
X_test = load_images("t10k-images.idx3-ubyte")
y_test = load_labels("t10k-labels.idx1-ubyte")

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)


Train: (60000, 28, 28) (60000,)
Test: (10000, 28, 28) (10000,)


## 2. 정규화 및 벡터 변환

In [2]:
X_train = X_train / 255.0
X_test = X_test / 255.0

# CNN용: (28, 28, 1), MLP용: 벡터화
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)


## 3. 기본 모델: KNN, Decision Tree

In [3]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

models_basic = {
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Decision Tree": DecisionTreeClassifier()
}

for name, model in models_basic.items():
    model.fit(X_train_scaled[:10000], y_train[:10000])  # 속도 이슈 대비 10000개만 학습
    preds = model.predict(X_test_scaled)
    print(f"\n{name} Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(classification_report(y_test, preds))


C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 지정된 파일을 찾을 수 없습니다
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Program Files\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Program Files\Python310\lib\subprocess.py", line 969, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Program Files\Python310\lib\subprocess.py", line 1438, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,



KNN Accuracy: 0.9127
              precision    recall  f1-score   support

           0       0.90      0.97      0.94       980
           1       0.92      0.99      0.95      1135
           2       0.93      0.88      0.91      1032
           3       0.89      0.93      0.91      1010
           4       0.93      0.90      0.91       982
           5       0.90      0.87      0.89       892
           6       0.95      0.94      0.95       958
           7       0.91      0.89      0.90      1028
           8       0.92      0.84      0.88       974
           9       0.88      0.88      0.88      1009

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000


Decision Tree Accuracy: 0.8027
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       980
           1       0.91      0.92      0.91      1135
           2       0.75 

## 4. 고급 모델: SVM, Random Forest, XGBoost

In [4]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models_advanced = {
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(n_estimators=50, use_label_encoder=False, eval_metric='mlogloss')
}

for name, model in models_advanced.items():
    print(f"\n{name} 학습 중...")
    model.fit(X_train_scaled[:10000], y_train[:10000])  # 속도 단축
    preds = model.predict(X_test_scaled)
    print(f"{name} Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(classification_report(y_test, preds))



SVM 학습 중...
SVM Accuracy: 0.9389
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       980
           1       0.98      0.99      0.98      1135
           2       0.91      0.94      0.93      1032
           3       0.94      0.94      0.94      1010
           4       0.94      0.95      0.94       982
           5       0.95      0.91      0.93       892
           6       0.95      0.95      0.95       958
           7       0.87      0.94      0.90      1028
           8       0.93      0.91      0.92       974
           9       0.95      0.90      0.92      1009

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000


Random Forest 학습 중...
Random Forest Accuracy: 0.9490
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       980
           1       0.98      0.99      0.99 

C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\xgboost\training.py:183: UserWarning: [14:59:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.9512
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       980
           1       0.98      0.98      0.98      1135
           2       0.94      0.94      0.94      1032
           3       0.94      0.95      0.94      1010
           4       0.95      0.95      0.95       982
           5       0.96      0.93      0.95       892
           6       0.95      0.96      0.95       958
           7       0.96      0.93      0.95      1028
           8       0.94      0.94      0.94       974
           9       0.93      0.94      0.93      1009

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.95      0.95      0.95     10000



## 5. 딥러닝 CNN 모델

In [5]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# GPU 사용 시 충돌 방지를 위해 CPU 강제 설정 (필요 시)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train_cnn, y_train_cat, epochs=5, batch_size=512, validation_split=0.1)


Epoch 1/5


: 

## 6. 성능 시각화 및 Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

# CNN 정확도 곡선
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# Confusion matrix for CNN
y_pred_cnn = model.predict(X_test_cnn).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred_cnn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
disp.plot(cmap='Blues')
plt.title("Confusion Matrix - CNN")
plt.show()
